In [0]:
%pip install gradio scikit-learn

In [0]:
dbutils.library.restartPython()

In [0]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [0]:
# Load and prepare the dataset
df = pd.read_csv('/Volumes/workspace/default/loan/Data.csv')

# Drop rows with missing target variable
df = df.dropna(subset=['Loan_Status'])

# Fill missing values
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

In [0]:
# Create derived features
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['TotalIncome_sqrt'] = np.sqrt(df['TotalIncome'])
df['LoanAmount_sqrt'] = np.sqrt(df['LoanAmount'])

# Encode categorical variables
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Married'] = df['Married'].map({'Yes': 1, 'No': 0})
df['Education'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})
df['Property_Area'] = df['Property_Area'].map({'Urban': 2, 'Semiurban': 1, 'Rural': 0})
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(float)
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

print("Features engineered and encoded")

In [0]:
# Select features for model
feature_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
                'Credit_History', 'Property_Area', 'TotalIncome_sqrt', 'LoanAmount_sqrt']

X = df[feature_cols]
y = df['Loan_Status']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_scaled, y)

accuracy = model.score(X_scaled, y)
print(f"Model trained with accuracy: {accuracy:.2%}")

In [0]:
def predict_loan_eligibility(gender, married, dependents, education, self_employed, 
                             applicant_income, coapplicant_income, loan_amount, 
                             loan_term, credit_history, property_area):
    """
    Predict loan eligibility based on applicant details
    """
    try:
        # Encode inputs
        gender_enc = 1 if gender == "Male" else 0
        married_enc = 1 if married == "Yes" else 0
        dependents_enc = 3 if dependents == "3+" else int(dependents)
        education_enc = 1 if education == "Graduate" else 0
        self_employed_enc = 1 if self_employed == "Yes" else 0
        property_area_enc = {"Urban": 2, "Semiurban": 1, "Rural": 0}[property_area]
        credit_history_enc = float(credit_history)
        
        # Calculate derived features
        total_income = float(applicant_income) + float(coapplicant_income)
        total_income_sqrt = np.sqrt(total_income)
        loan_amount_sqrt = np.sqrt(float(loan_amount))
        
        # Create feature array
        features = np.array([[gender_enc, married_enc, dependents_enc, education_enc, 
                            self_employed_enc, credit_history_enc, property_area_enc,
                            total_income_sqrt, loan_amount_sqrt]])
        
        # Scale features
        features_scaled = scaler.transform(features)
        
        # Predict
        prediction = model.predict(features_scaled)[0]
        probability = model.predict_proba(features_scaled)[0]
        
        # Format result
        if prediction == 1:
            result = "✅ LOAN APPROVED"
            confidence = f"Confidence: {probability[1]:.1%}"
            color = "green"
        else:
            result = "❌ LOAN REJECTED"
            confidence = f"Confidence: {probability[0]:.1%}"
            color = "red"
        
        return f"<h2 style='color: {color};'>{result}</h2><p style='font-size: 18px;'>{confidence}</p>"
    
    except Exception as e:
        return f"<h2 style='color: orange;'>⚠️ Error</h2><p>{str(e)}</p>"

print("Prediction function defined")

In [0]:
# Create Gradio interface
with gr.Blocks(title="Loan Eligibility Predictor", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🏦 Loan Eligibility Prediction System
        ### Enter applicant details below to check loan eligibility
        """
    )
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Personal Information")
            gender = gr.Dropdown(["Male", "Female"], label="Gender", value="Male")
            married = gr.Dropdown(["Yes", "No"], label="Marital Status", value="Yes")
            dependents = gr.Dropdown(["0", "1", "2", "3+"], label="Number of Dependents", value="0")
            education = gr.Dropdown(["Graduate", "Not Graduate"], label="Education", value="Graduate")
            self_employed = gr.Dropdown(["Yes", "No"], label="Self Employed", value="No")
            property_area = gr.Dropdown(["Urban", "Semiurban", "Rural"], label="Property Area", value="Urban")
        
        with gr.Column():
            gr.Markdown("### Financial Information")
            applicant_income = gr.Number(label="Applicant Income (₹)", value=5000)
            coapplicant_income = gr.Number(label="Coapplicant Income (₹)", value=0)
            loan_amount = gr.Number(label="Loan Amount (₹ thousands)", value=150)
            loan_term = gr.Number(label="Loan Term (months)", value=360)
            credit_history = gr.Dropdown(["1.0", "0.0"], label="Credit History (1=Good, 0=Bad)", value="1.0")
    
    predict_btn = gr.Button("🔮 Predict Eligibility", variant="primary", size="lg")
    
    output = gr.HTML(label="Prediction Result")
    
    predict_btn.click(
        fn=predict_loan_eligibility,
        inputs=[gender, married, dependents, education, self_employed, 
                applicant_income, coapplicant_income, loan_amount, 
                loan_term, credit_history, property_area],
        outputs=output
    )
    
    gr.Markdown(
        """
        ---
        **Note:** This is a machine learning prediction based on historical data. 
        Actual loan approval depends on additional factors and bank policies.
        """
    )

# Launch the interface with public sharing enabled
demo.launch(share=True, inline=True)